# Manhwa Studio — Renderizador CPU no Google Colab
Este notebook inicia uma API temporária de renderização com FFmpeg. **Não precisa editar código**: conecte o ambiente e execute as células em ordem. Ao final, copie a URL HTTPS exibida para conectar o site (a integração do endereço no site é uma etapa separada). Mantenha esta aba aberta durante a renderização.

Use somente vídeos e imagens que você tem direito de processar. A sessão e os arquivos temporários são apagados quando o Colab encerra.

In [ ]:
# 1) Instalar FFmpeg, Cloudflared e dependências da API
!apt-get -qq update
!apt-get -qq install -y ffmpeg curl
!pip -q install fastapi uvicorn python-multipart

In [ ]:
# 2) Baixar o Cloudflare Tunnel para criar uma URL HTTPS temporária
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
%%writefile /content/manhwa_cpu_api.py
import os, secrets, subprocess, tempfile, threading
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.background import BackgroundTask

TOKEN = os.environ.get('MANHWA_RENDER_TOKEN', '')
app = FastAPI(title='Manhwa Studio Colab CPU Renderer')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=False, allow_methods=['GET','POST'], allow_headers=['*'])
LIMIT = 7 * 1024**3
slots = threading.BoundedSemaphore(1)

def cleanup(folder):
    import shutil; shutil.rmtree(folder, ignore_errors=True)

@app.get('/health')
def health(): return {'status':'ok','renderer':'ffmpeg-cpu','hardware_acceleration':False}

@app.post('/render')
async def render(video: UploadFile=File(...), background: UploadFile=File(...), format: str=Form('16:9'), resolution: str=Form('1080p'), zoom: float=Form(100), offset_x: float=Form(0), offset_y: float=Form(0), crop_to_fill: bool=Form(True), hide_watermark: bool=Form(False), mark_x: float=Form(80), mark_y: float=Form(85), mark_w: float=Form(15), mark_h: float=Form(8), x_render_token: str=Header(default='')):
    if not TOKEN or not secrets.compare_digest(x_render_token, TOKEN): raise HTTPException(401, 'Token de renderização inválido.')
    if not slots.acquire(blocking=False): raise HTTPException(429, 'O Colab já está renderizando outro vídeo.')
    folder=tempfile.mkdtemp(prefix='manhwa-render-'); root=Path(folder)
    try:
        vp=root/'input-video'; bp=root/'frame-image'; out=root/'render-final.mp4'
        for upload,path in ((video,vp),(background,bp)):
            size=0
            with path.open('wb') as f:
                while True:
                    chunk=await upload.read(8*1024*1024)
                    if not chunk: break
                    size+=len(chunk)
                    if size>LIMIT: raise HTTPException(413,'Arquivo excede 7 GB.')
                    f.write(chunk)
        if format not in ('16:9','9:16') or resolution not in ('1080p','4K'): raise HTTPException(400,'Formato ou resolução inválidos.')
        w,h=((3840,2160) if resolution=='4K' else (1920,1080)) if format=='16:9' else ((2160,3840) if resolution=='4K' else (1080,1920))
        safe_w,safe_h=w*.84,h*.84
        scale_mode='increase' if crop_to_fill else 'decrease'
        vf=(f"[1:v]scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2:black[bg];"
            f"[0:v]scale=w='if(gte(a,{safe_w/safe_h}),{safe_w},-2)':h='if(gte(a,{safe_w/safe_h}),-2,{safe_h})':force_original_aspect_ratio={scale_mode},scale=iw*{max(.1,min(4,zoom/100))}:ih*{max(.1,min(4,zoom/100))}[scaled];"
            f"[bg][scaled]overlay=x='(W-w)/2+{max(-100,min(100,offset_x))}*{safe_w/200}':y='(H-h)/2+{max(-100,min(100,offset_y))}*{safe_h/200}':shortest=1[base]")
        if hide_watermark:
            # Desfoque retangular na região percentual indicada do vídeo original, projetada na área do vídeo.
            mx,my,mw,mh=[max(0,min(100,float(v))) for v in (mark_x,mark_y,mark_w,mark_h)]
            # O desfoque é aplicado no quadro inteiro da composição para evitar recortes inválidos.
            vf += f";[base]boxblur=10:1[blur];[base][blur]blend=all_expr='if(between(X,{int(w*mx/100)},{int(w*min(100,mx+mw)/100)})*between(Y,{int(h*my/100)},{int(h*min(100,my+mh)/100)}),B,A)'[vout]"
            vmap='[vout]'
        else: vmap='[base]'
        cmd=['ffmpeg','-y','-i',str(vp),'-loop','1','-i',str(bp),'-filter_complex',vf,'-map',vmap,'-map','0:a?','-c:v','libx264','-preset','veryfast','-crf','18','-pix_fmt','yuv420p','-c:a','aac','-b:a','192k','-movflags','+faststart','-shortest',str(out)]
        proc=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
        if proc.returncode!=0: raise HTTPException(500,'FFmpeg falhou: '+proc.stderr[-1800:])
        return FileResponse(out,media_type='video/mp4',filename='manhwa-studio-render.mp4',background=BackgroundTask(cleanup,folder))
    except Exception:
        cleanup(folder); raise
    finally: slots.release()


In [ ]:
# 3) Iniciar API + túnel público HTTPS. Execute esta célula por último.
import os, secrets, subprocess, time, re, threading
from IPython.display import display, Markdown
TOKEN = secrets.token_urlsafe(32)
os.environ['MANHWA_RENDER_TOKEN'] = TOKEN
api = subprocess.Popen(['uvicorn','manhwa_cpu_api:app','--host','0.0.0.0','--port','8000'],cwd='/content',stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
public_url = None
deadline=time.time()+90
while time.time()<deadline and tunnel.poll() is None:
    line=tunnel.stdout.readline()
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
    if match: public_url=match.group(0); break
    time.sleep(.2)
if not public_url: raise RuntimeError('Não foi possível criar o túnel. Interrompa e execute novamente esta célula.')
display(Markdown('## Renderizador CPU conectado\n**URL temporária:** `'+public_url+'`\n\n**Token de sessão:** `'+TOKEN+'`\n\n**Health:** `'+public_url+'/health`\n\nMantenha esta célula em execução. A URL e o token mudam quando uma nova sessão é iniciada.'))
print('Servidor iniciado. Não feche esta aba durante o processamento.')

## Como usar
1. Abra o notebook pelo link do repositório no Colab.
2. Clique em **Conectar**.
3. Execute as células 1, 2, 3 e 4, nesta ordem (Shift+Enter).
4. A última célula mostrará a URL e o token temporários.
5. Mantenha o notebook conectado durante o render.

**Importante:** este notebook disponibiliza a API de renderização. O site ainda precisa enviar os arquivos e as opções para `/render` usando o cabeçalho `X-Render-Token`. Não compartilhe a URL/token; eles dão acesso à sessão. O processamento usa `libx264` em CPU e pode demorar mais que a renderização acelerada por GPU.

In [ ]:
# Opcional: parar os processos ao terminar
for p in (api, tunnel):
    try: p.terminate()
    except Exception: pass
print('Servidor e túnel encerrados.')